# CNN Baseline Experiment on MNIST

This notebook implements the classical CNN baseline for MNIST image classification.

Recommended use:
1. Open this notebook in Google Colab from GitHub.
2. Select **Runtime → Disconnect and delete runtime**.
3. Select **Runtime → Run all**.
4. Verify that the dataset is downloaded automatically and the metrics are produced without requiring Google Drive files.


In [ ]:
# CELL 1 : Install dependencies (if not installed)
!pip install torch torchvision scikit-image medmnist --quiet

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
from tqdm import tqdm
import time
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay

# pilih device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
# CELL 2 : Load MNIST stratified subset (1200/train/kelas, 200/test/kelas)
from collections import defaultdict
import numpy as np
from torchvision import datasets, transforms
import torch

def load_mnist_subset(batch_size=64,
                      train_per_class=1200,
                      test_per_class=200,
                      img_size=(22,22),
                      root='./data'):
    # transformasi
    transform = transforms.Compose([
        transforms.Grayscale(num_output_channels=1),
        transforms.Resize(img_size),
        transforms.ToTensor(),
    ])

    # unduh full MNIST
    full_train = datasets.MNIST(root=root, train=True,  transform=transform, download=True)
    full_test  = datasets.MNIST(root=root, train=False, transform=transform, download=True)

    def stratified_subset(dataset, samples_per_class):
        class_idxs = defaultdict(list)
        for idx, (_, lbl) in enumerate(dataset):
            class_idxs[lbl].append(idx)
        selected = []
        for lbl, idxs in class_idxs.items():
            if samples_per_class > len(idxs):
                raise ValueError(f"Permintaan {samples_per_class} sampel untuk kelas {lbl}, "
                                 f"padahal hanya ada {len(idxs)} data.")
            chosen = np.random.choice(idxs, samples_per_class, replace=False)
            selected.extend(chosen.tolist())
        return torch.utils.data.Subset(dataset, sorted(selected))

    train_subset = stratified_subset(full_train, train_per_class)
    test_subset  = stratified_subset(full_test,  test_per_class)

    train_loader = torch.utils.data.DataLoader(
        train_subset, batch_size=batch_size, shuffle=True,  num_workers=0
    )
    test_loader  = torch.utils.data.DataLoader(
        test_subset,  batch_size=batch_size, shuffle=False, num_workers=0
    )
    return train_loader, test_loader

# (Opsional) sanity check ukuran data
tmp_tr, tmp_te = load_mnist_subset(batch_size=64, train_per_class=1200, test_per_class=200)
print(f"Train batches: {len(tmp_tr)}, Test batches: {len(tmp_te)}")
del tmp_tr, tmp_te

In [ ]:
# CELL 3 : Classical CNN baseline (Model 1 di QFE paper)

# CHANNEL-POOLING UTILITY
def channel_max_pool2d(x, kernel_size=2):
    b, c, h, w = x.size()
    if c % kernel_size != 0:
        raise ValueError(f"Channel count {c} not divisible by pool size {kernel_size}.")
    x = x.view(b, c // kernel_size, kernel_size, h, w)
    x, _ = x.max(dim=2)
    return x

class ClassicalCNN1(nn.Module):
    def __init__(self):
        super().__init__()
        # — Block 1: Conv1(1→18), stride=2 → spatial 22×22→11×11,
        #   then channel_max_pool2d(18→9) —
        self.conv1 = nn.Conv2d(1, 18, kernel_size=3, stride=2, padding=1)

        # — Block 2: Conv2(9→54), stride=2 → spatial 11×11→6×6,
        #   then channel_max_pool2d(54→27) —
        self.conv2 = nn.Conv2d(9, 54, kernel_size=3, stride=2, padding=1)

        # Hitung otomatis dim input FC = 27 channels × 6 × 6 = 972
        with torch.no_grad():
            x = torch.zeros(1, 1, 22, 22)
            x = F.relu(self.conv1(x))                   # → (1,18,11,11)
            x = channel_max_pool2d(x, 2)                # → (1,9,11,11)
            x = F.relu(self.conv2(x))                   # → (1,54,6,6)
            x = channel_max_pool2d(x, 2)                # → (1,27,6,6)
            print("x.shape:", x.shape)                  # ⬅️ cek bentuk
            print("FC input size:", x.numel())
            fc_input = x.numel()                        # 27*6*6 = 972

        # FC head: 972 → 128 → 64 → 2 (binary)
        self.fc1 = nn.Linear(fc_input, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)   # ⬅️ 10 kelas: MNIST

    def forward(self, x):
        # Block 1
        x = F.relu(self.conv1(x))                     # b×18×11×11
        x = channel_max_pool2d(x, 2)                  # b×9×11×11

        # Block 2
        x = F.relu(self.conv2(x))                     # b×54×6×6
        x = channel_max_pool2d(x, 2)                  # b×27×6×6

        # Flatten & FC
        x = x.view(x.size(0), -1)                     # b×972
        x = F.relu(self.fc1(x))                       # b×128
        x = F.relu(self.fc2(x))                       # b×64
        return self.fc3(x)                            # b×10


In [ ]:
# CELL 4 : Training berbasis Iteration (bukan Epoch) — style sama, dataset = MNIST
from sklearn.metrics import roc_auc_score  # ⬅️ tambah untuk AUC

train_loader, test_loader = load_mnist_subset(batch_size=64, train_per_class=1200, test_per_class=200)
model     = ClassicalCNN1().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.0005)
loss_fn   = nn.CrossEntropyLoss()

# Hitung parameter
num_params = sum(p.numel() for p in model.parameters())
num_trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total Parameters: {num_params}")
print(f"Trainable Parameters: {num_trainable_params}")

# Fungsi evaluasi (tanpa perubahan gaya/metode)
def evaluate(model, data_loader):
    loss_sum, correct, total = 0.0, 0, 0
    model.eval()
    with torch.no_grad():
        for images, labels in data_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)                      # → (b, num_classes=2)
            loss = loss_fn(outputs, labels)
            loss_sum += loss.item()
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return loss_sum / len(data_loader), correct / total

# Loop training berbasis iteration (pengukuran waktu & warna grafik dipertahankan)
def train_iterations(model,
                     train_loader,
                     test_loader,
                     max_iterations=1000,     # ⬅️ 1000 iterasi sesuai konversi 5 epoch
                     eval_interval=1000):

    start_time    = time.time()
    fc_time_total = 0.0
    train_losses, train_accs = [], []
    test_losses,  test_accs  = [], []

    loader_iter = iter(train_loader)
    pbar = tqdm(total=max_iterations,
                desc=f"Iter 0/{max_iterations}",
                ncols=100, leave=True)

    for iteration in range(1, max_iterations + 1):
        pbar.set_description(f"Iter {iteration}/{max_iterations}")
        try:
            images, labels = next(loader_iter)
        except StopIteration:
            loader_iter = iter(train_loader)
            images, labels = next(loader_iter)
        images, labels = images.to(device), labels.to(device)

        # ─── TRAIN STEP ─────────────────────────
        model.train()
        optimizer.zero_grad()

        # 1) Forward sampai sebelum FC
        x = F.relu(model.conv1(images))
        x = channel_max_pool2d(x, 2)
        x = F.relu(model.conv2(x))
        x = channel_max_pool2d(x, 2)
        feats = x.view(x.size(0), -1)

        # 2) Timing FC-only
        t0 = time.perf_counter()
        h  = F.relu(model.fc1(feats))
        h  = F.relu(model.fc2(h))
        outputs = model.fc3(h)
        fc_time_total += time.perf_counter() - t0

        # 3) Backprop + update
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()

        # ─── HITUNG METRIK BATCH ───────────────
        train_losses.append(loss.item())
        batch_acc = (outputs.argmax(1) == labels).float().mean().item()
        train_accs.append(batch_acc)

        pbar.set_postfix({"Loss": f"{loss.item():.4f}",
                          "Acc":  f"{batch_acc:.4f}"})
        pbar.update(1)

        # ─── EVALUASI PERIODIK ─────────────────
        if iteration % eval_interval == 0 or iteration == max_iterations:
            tl, ta = evaluate(model, test_loader)
            test_losses.append(tl)
            test_accs.append(ta)
            tqdm.write(f"→ Eval @Iter {iteration}: TestLoss={tl:.4f}, TestAcc={ta:.4f}")

    pbar.close()
    total_time = time.time() - start_time
    print(f"\nTotal training time                 : {total_time:.4f}s")
    print(f"Total FC-layer computation time     : {fc_time_total:.4f}s")

    # Kumpulkan prediksi & probabilitas utk AUC (multiclass, 10 kelas)
    all_preds, all_labels, all_scores = [], [], []
    model.eval()
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            logits = model(images)                               # (B, 10)
            preds  = logits.argmax(dim=1).cpu().tolist()
            probs  = F.softmax(logits, dim=1).cpu().numpy()      # (B, 10)

            all_preds.extend(preds)
            all_scores.extend(probs.tolist())                    # list of length-10
            all_labels.extend(labels.tolist())                   # int labels 0..9

    prec = precision_score(all_labels, all_preds, average='macro', zero_division=0)
    rec  = recall_score(all_labels, all_preds, average='macro',   zero_division=0)
    f1   = f1_score(all_labels, all_preds, average='macro',       zero_division=0)
    cm   = confusion_matrix(all_labels, all_preds)

    # ROC-AUC MULTICLASS (macro, OVR)
    import numpy as np
    try:
        auc = roc_auc_score(all_labels, np.array(all_scores), multi_class='ovr', average='macro')
    except ValueError:
        auc = float('nan')

    print(f"\nFinal Test Precision: {prec:.4f}")
    print(f"Final Test Recall   : {rec:.4f}")
    print(f"Final Test F1-score : {f1:.4f}")
    print(f"Final Test ROC-AUC  : {auc:.4f}")

    # Confusion Matrix (judul disesuaikan dataset)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                  display_labels=list(range(cm.shape[0])))
    fig, ax = plt.subplots(figsize=(6,6))
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_xlabel('Predicted label')
    ax.set_ylabel('True label')
    ax.set_title('CNN - MNIST')  # ⬅️ judul baru
    plt.show()

    # Siapkan sumbu (tanpa perubahan)
    x_iter = list(range(1, max_iterations + 1))
    x_test = list(range(eval_interval, max_iterations+1, eval_interval))
    if x_test and x_test[-1] != max_iterations:
        x_test.append(max_iterations)

    # Grafik 1: Train/Test Loss vs Iteration
    plt.figure(figsize=(6,4))
    plt.plot(x_iter, train_losses, label='Train Loss')
    plt.plot(x_test, test_losses, label='Test Loss')
    plt.xlabel('Iteration'); plt.ylabel('Loss'); plt.ylim(0, 3)
    plt.legend(); plt.title('Loss vs Iteration'); plt.show()

    # Grafik 2: Train/Test Accuracy vs Iteration
    plt.figure(figsize=(6,4))
    plt.plot(x_iter, train_accs, label='Train Acc')
    plt.plot(x_test, test_accs, label='Test Acc')
    plt.xlabel('Iteration'); plt.ylabel('Accuracy'); plt.ylim(0, 1)
    plt.legend(); plt.title('Accuracy vs Iteration'); plt.show()

    # Grafik 3: gabungan TRAIN (loss & acc) — warna dipertahankan
    fig, ax1 = plt.subplots(figsize=(6,4))
    ax2 = ax1.twinx()
    ax1.plot(x_iter, train_losses, color='tab:orange', label='Batch Loss')
    ax2.plot(x_iter, train_accs,  color='tab:green',  label='Batch Acc')
    ax1.set_xlabel('Iteration'); ax1.set_ylabel('Loss'); ax2.set_ylabel('Accuracy')
    ax1.set_ylim(0,3);                 ax2.set_ylim(0,1)
    l1,lab1 = ax1.get_legend_handles_labels()
    l2,lab2 = ax2.get_legend_handles_labels()
    ax1.legend(l1 + l2, lab1 + lab2, loc='center right', framealpha=0.8)
    plt.title('Train Performance'); plt.show()

    # Grafik 4: gabungan TEST (loss & acc) — warna dipertahankan
    fig, ax1 = plt.subplots(figsize=(6,4))
    ax2 = ax1.twinx()
    ax1.plot(x_test, test_losses, color='tab:orange', label='Test Loss')
    ax2.plot(x_test, test_accs,  color='tab:green',  label='Test Acc')
    ax1.set_xlabel('Iteration'); ax1.set_ylabel('Loss'); ax2.set_ylabel('Accuracy')
    ax1.set_ylim(0,3);                 ax2.set_ylim(0,1)
    l1,lab1 = ax1.get_legend_handles_labels()
    l2,lab2 = ax2.get_legend_handles_labels()
    ax1.legend(l1 + l2, lab1 + lab2, loc='center right', framealpha=0.8)
    plt.title('Test Performance'); plt.show()

In [ ]:
# CELL 5 : Run training (pokoknya harus 5 epoch)
train_iterations(model, train_loader, test_loader,
                 max_iterations=1000,
                 eval_interval=1000)

In [ ]:
# CELL 6 : Menyimpan bobot model yang sudah terlatih
def save_model(model, filename="CNN_breastmnist.pth"):
    torch.save(model.state_dict(), filename)
    print(f"Model state_dict saved to {filename}")

# Panggil setelah training selesai
save_model(model)
